In [ ]:
import os
import pandas as pd
import geopandas as gpd
import numpy as np
import pyproj
import libpysal
from esda.getisord import G_Local
from pathlib import Path

In [ ]:
BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == "notebooks" else BASE_DIR

# --- Grid geometry (Step 5 output) ---
grid_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_grid_2_5km_wgs84.gpkg")
grid = gpd.read_file(grid_path)  # columns: Cell_ID, cell_size_km, geometry

In [ ]:
# --- Cell x Year wide table (Step 7 output) ---
wide_path = os.path.join(PROJECT_ROOT, "data", "raw", "ihr_cell_year_wide.csv")
wide = pd.read_csv(wide_path)  # columns: Cell_ID, 2001, 2002, ..., 2025

year_cols = [c for c in wide.columns if c != "Cell_ID"]
print(f"Grid cells: {len(grid):,} | Wide table cells: {len(wide):,} | Years: {len(year_cols)}")

## Extract a small contiguous block for testing

In [ ]:
def extract_test_block(grid_gdf, n_side=6, cell_size_km=2.5, seed_cell_id=None):
    """
    Extract a roughly n_side x n_side contiguous block of cells from a regular
    grid, for quick local testing before running Gi* on the full grid.
    """
    gdf = grid_gdf.copy()
    if gdf.crs is None or gdf.crs.is_geographic:
        centroid = gdf.union_all().centroid
        laea_crs = pyproj.CRS.from_proj4(
            f"+proj=laea +lat_0={centroid.y} +lon_0={centroid.x} +datum=WGS84 +units=m +no_defs"
        )
        gdf = gdf.to_crs(laea_crs)

    gdf["cx"] = gdf.geometry.centroid.x
    gdf["cy"] = gdf.geometry.centroid.y

    if seed_cell_id is not None:
        seed = gdf.loc[gdf["Cell_ID"] == seed_cell_id].iloc[0]
    else:
        # Default: nearest cell to the overall grid centroid -> safest bet
        # for landing in an unbroken interior block (away from boundary edges)
        gx0, gy0 = gdf["cx"].mean(), gdf["cy"].mean()
        gdf["_dist"] = (gdf["cx"] - gx0) ** 2 + (gdf["cy"] - gy0) ** 2
        seed = gdf.loc[gdf["_dist"].idxmin()]

    cx0, cy0 = seed["cx"], seed["cy"]
    half_extent = (n_side / 2) * cell_size_km * 1000

    block = gdf[
        (gdf["cx"] >= cx0 - half_extent) & (gdf["cx"] < cx0 + half_extent) &
        (gdf["cy"] >= cy0 - half_extent) & (gdf["cy"] < cy0 + half_extent)
    ].copy()

    return block.drop(columns=["cx", "cy", "_dist"], errors="ignore")




In [ ]:
N_SIDE = 6  # change to 20 for a bigger test block
test_grid = extract_test_block(grid, n_side=N_SIDE, cell_size_km=2.5)
print(f"Test block: {len(test_grid)} cells (target ~{N_SIDE**2})")

# Keep only the years/cells present in both the grid subset and the wide table
test_wide = wide[wide["Cell_ID"].isin(test_grid["Cell_ID"])].copy()
test_grid = test_grid[test_grid["Cell_ID"].isin(test_wide["Cell_ID"])].reset_index(drop=True)
print(f"After matching to loss data: {len(test_grid)} cells")

## Spatial weights + Gi* per year (this loop is the pattern for your later per-year production run)

In [ ]:
test_wide.set_index("Cell_ID")[year_cols].describe()

In [ ]:
print("Total loss by year:")
print(wide[year_cols].sum())

print("\nNumber of cells with any loss:")
print((wide[year_cols].sum(axis=1) > 0).sum())

print("\nMaximum loss across all cells and years:")
print(wide[year_cols].max().max())

In [ ]:
# Find the cell with the greatest total loss across 2001–2025
loss_sum = wide[year_cols].sum(axis=1)

seed_cell = wide.loc[loss_sum.idxmax(), "Cell_ID"]

print("Seed Cell_ID:", seed_cell)
print("Total 2001–2025 loss:", loss_sum.max())

In [ ]:
# Extract a larger test neighbourhood around that known-loss cell
N_SIDE = 10

test_grid = extract_test_block(
    grid,
    n_side=N_SIDE,
    cell_size_km=2.5,
    seed_cell_id=seed_cell
)

# Keep only cells available in the forest loss dataset
test_grid = test_grid[
    test_grid["Cell_ID"].isin(wide["Cell_ID"])
].reset_index(drop=True)

test_wide = wide[
    wide["Cell_ID"].isin(test_grid["Cell_ID"])
].copy()

print(f"Test block: {len(test_grid)} cells")

In [ ]:
print(test_wide[year_cols].describe())print("Total loss in test block by year:")
print(test_wide[year_cols].sum())

print("\nCells with any loss in test block:")
print((test_wide[year_cols].sum(axis=1) > 0).sum())

In [ ]:
print("Total loss in test block by year:")
print(test_wide[year_cols].sum())

print("\nCells with any loss in test block:")
print((test_wide[year_cols].sum(axis=1) > 0).sum())

In [ ]:

#w stores all those relationship of Queen Cells with X
w = libpysal.weights.Queen.from_dataframe(test_grid, ids=test_grid["Cell_ID"].tolist())
#This makes the neighbour weights for each cell sum to 1.
'''original: each neighbour = 1

after row-standardization: each neighbour = 1/8 = 0.125
or each neighbour = 1/5 = 0.20'''
w.transform = "r"

results = []
for year in year_cols:
    y = test_wide.set_index("Cell_ID").loc[w.id_order, year].values

    gi = G_Local(y, w, star=True, permutations=99)  # bump permutations for the real run

    for cid, z, p_sim, p_norm in zip(w.id_order, gi.Zs, gi.p_sim, gi.p_norm):
        results.append({
            "Cell_ID": cid,
            "Year": int(year),
            "Gi_star_z": z,
            "Gi_star_p_sim": p_sim,
            "Gi_star_p_norm": p_norm,
        })

gi_test_df = pd.DataFrame(results)
print(f"Gi* test results: {len(gi_test_df):,} rows ({gi_test_df['Cell_ID'].nunique()} cells x {gi_test_df['Year'].nunique()} years)")
gi_test_df.head(10)

### Quick look — one cell's trajectory, one year's snapshot

In [ ]:
# Trace a single cell across years (visually check it's not all zeros/NaNs)
sample_cell = gi_test_df["Cell_ID"].iloc[0]
print(gi_test_df[gi_test_df["Cell_ID"] == sample_cell][["Year", "Gi_star_z", "Gi_star_p_sim"]])

# Snapshot for one year across the test block
sample_year = int(year_cols[0])
print(gi_test_df[gi_test_df["Year"] == sample_year].sort_values("Gi_star_z", ascending=False).head())

In [ ]:
for year in ["2001", "2010", "2015", "2020", "2025"]:
    vals = test_wide.set_index("Cell_ID")[year]

    print(
        year,
        "mean =", vals.mean(),
        "max =", vals.max(),
        "nonzero =", (vals > 0).sum(),
        "cells =", len(vals)
    )

In [ ]:
# Show the highest-loss cells in the test neighbourhood
for year in ["2001", "2010", "2015", "2020", "2025"]:
    print(f"\n{year}")
    print(
        test_wide[["Cell_ID", year]]
        .sort_values(year, ascending=False)
        .head(10)
        .to_string(index=False)
    )

In [ ]:
for year in ["2001", "2010", "2015", "2020", "2025"]:
    print(f"\n{year}")

    print(
        gi_test_df[
            gi_test_df["Year"] == int(year)
        ][
            ["Cell_ID", "Gi_star_z", "Gi_star_p_sim"]
        ]
        .sort_values("Gi_star_z", ascending=False)
        .head(10)
        .to_string(index=False)
    )

In [ ]:
year = 2001

plot_df = test_grid.merge(
    gi_test_df[gi_test_df["Year"] == year],
    on="Cell_ID",
    how="left"
)

plot_df.plot(
    column="Gi_star_z",
    legend=True,
    figsize=(8, 8)
)